
# NanoVision — Drug Screening Dashboard (Colab)

This notebook replicates the **Drug Screening Dashboard** (`src/pages/Screening.tsx`) in Python with all key features:
- Analysis history loading (from JSON upload) and mock history generation
- Search/filter + multi-select sample picking
- Delete selected samples
- Risk / Final / PSNR comparison bar chart
- Full section drill-down for all dashboard modules:
  1. Screening Module Overview
  2. Morphology-Based Screening
  3. Reconstruction Quality Assessment
  4. Nano–Bio Interaction Indicators
  5. Formulation Stability Evaluation
  6. Drug Prediction (Multimodal)
  7. Aggregation Behavior Analysis
  8. Multi-Factor Risk Score Calculation
  9. Screening Decision Classification
  10. Model-Based Screening Head (NanoVisionNet-X)
  11. Screening Output Visualization


In [ ]:

#@title 1) Install dependencies (Colab)
!pip -q install ipywidgets matplotlib pandas numpy


In [ ]:

#@title 2) Imports
import base64
import io
import json
import math
import random
import uuid
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, Markdown, clear_output, display

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


In [ ]:

#@title 3) Full analysis mock engine (mirrors src/lib/mockAnalysis.ts fields)
def clamp(value, min_value=0, max_value=100):
    return max(min_value, min(max_value, value))

MOCK_COMPOUNDS = [
    {"smiles": "CC(=O)OC1=CC=CC=C1C(=O)O", "molecularWeight": 180.16, "solubility": 3.2},
    {"smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "molecularWeight": 206.28, "solubility": 0.08},
    {"smiles": "CC(=O)NC1=CC=C(O)C=C1", "molecularWeight": 151.16, "solubility": 14.0},
    {"smiles": "CN1CCC[C@H]1C2=CN=CC=C2", "molecularWeight": 162.23, "solubility": 10.5},
    {"smiles": "C1=CC=C(C=C1)C=O", "molecularWeight": 106.12, "solubility": 6.7},
]

def run_mock_analysis(seed=None):
    if seed is not None:
        random.seed(seed)

    nuclei_count = random.randint(30, 149)
    mean_area = round(random.random() * 500 + 200, 1)
    std_area = round(random.random() * 100 + 20, 1)
    circularity = round(random.random() * 0.4 + 0.6, 3)
    aggregation_score = round(random.random() * 0.6 + 0.1, 2)
    dice_score = round(random.random() * 0.15 + 0.82, 3)
    iou_score = round(dice_score - random.random() * 0.1, 3)
    density_per_unit = round(nuclei_count / (random.random() * 5 + 8), 1)
    stability_score = round(random.random() * 40 + 55, 1)
    uniformity_score = round(random.random() * 35 + 60, 1)
    interaction_strength = round(random.random() * 50 + 40, 1)

    total = stability_score + uniformity_score + (100 - aggregation_score * 100) + interaction_strength
    screening_decision = "Promising Candidate" if total > 300 else "Needs Optimization" if total > 220 else "Low Performance"

    particle_sizes = [
        {"size": "0-50", "count": random.randint(5, 24)},
        {"size": "50-100", "count": random.randint(15, 54)},
        {"size": "100-200", "count": random.randint(20, 49)},
        {"size": "200-400", "count": random.randint(10, 34)},
        {"size": "400-600", "count": random.randint(3, 17)},
        {"size": "600+", "count": random.randint(1, 8)},
    ]

    density_data = [
        {"region": "Q1", "density": random.randint(5, 34)},
        {"region": "Q2", "density": random.randint(5, 34)},
        {"region": "Q3", "density": random.randint(5, 34)},
        {"region": "Q4", "density": random.randint(5, 34)},
    ]

    dynamics_data = []
    for i, t in enumerate(["T0", "T1", "T2", "T3", "T4", "T5"]):
        dynamics_data.append({
            "t": t,
            "diffusion": round(clamp(42 + i * 8 + random.random() * 10), 1),
            "movement": round(clamp(34 + i * 9 + random.random() * 9), 1),
            "cellResponse": round(clamp(38 + i * 7 + random.random() * 8), 1),
        })

    risk_score = clamp(100 - (stability_score * 0.35 + uniformity_score * 0.25 + interaction_strength * 0.2 + (100 - aggregation_score * 100) * 0.2))
    multi_factor_score = clamp(stability_score * 0.4 + uniformity_score * 0.3 + interaction_strength * 0.3)
    area_comparison_score = clamp(100 - std_area * 0.7)
    aggregation_detection_score = clamp(aggregation_score * 100)
    psnr = round(22 + random.random() * 15, 2)
    ssim = round(0.75 + random.random() * 0.23, 3)
    structural_clarity = clamp((psnr - 20) * 5)
    segmentation_confidence = clamp((dice_score * 100 + iou_score * 100) / 2)
    membrane_interaction_score = clamp(interaction_strength)
    cytotoxicity_risk = clamp(aggregation_score * 90 + (1 - circularity) * 25)
    surface_stability = clamp(stability_score - aggregation_score * 15)
    zeta_potential_proxy = round(-35 + random.random() * 45, 1)
    diffusion_coefficient = round(0.15 + random.random() * 0.9, 3)
    transport_efficiency = clamp(uniformity_score * 0.6 + stability_score * 0.4)
    bioavailability_prediction = clamp(transport_efficiency * 0.65 + interaction_strength * 0.35)
    structural_consistency = clamp(100 - std_area * 0.5 + circularity * 15)
    cluster_formation = clamp(aggregation_score * 100)
    density_variation = clamp(abs(density_data[0]["density"] - density_data[2]["density"]) * 2.1)
    particle_overlap = clamp(aggregation_score * 80 + (100 - circularity * 100) * 0.25)
    stability_risk = clamp((100 - stability_score) * 0.6 + cluster_formation * 0.4)
    feature_vector_integration = clamp((segmentation_confidence + membrane_interaction_score + bioavailability_prediction) / 3)
    weighted_score = clamp(feature_vector_integration * 0.45 + (100 - risk_score) * 0.55)
    threshold_gap = round(weighted_score - 62, 2)
    final_screening_score = clamp(weighted_score - risk_score * 0.2)
    model_head_risk = clamp(100 / (1 + math.exp(-(risk_score - 45) / 8)))

    selected_compound = random.choice(MOCK_COMPOUNDS)
    smiles = selected_compound["smiles"]
    molecular_weight = round(selected_compound["molecularWeight"] + (random.random() * 6 - 3), 2)
    binding_affinity = round(-5.2 - random.random() * 6.4, 2)
    solubility = round(clamp(selected_compound["solubility"] + (random.random() * 2.4 - 1.2), 0.02, 20), 2)
    cell_uptake_rate = round(clamp(45 + random.random() * 45), 1)
    toxicity_label = "Low" if cytotoxicity_risk < 35 else "Moderate" if cytotoxicity_risk < 62 else "High"
    protein_interaction = round(clamp(interaction_strength + random.random() * 10), 1)
    target_receptor_binding = round(clamp(55 + random.random() * 35), 1)

    latest = dynamics_data[-1]
    diffusion_trend = round(latest["diffusion"], 1)
    movement_trend = round(latest["movement"], 1)
    response_trend = round(latest["cellResponse"], 1)
    formulation_transport_score = round(clamp((transport_efficiency + diffusion_trend) / 2), 1)
    predicted_drug_efficacy = round(clamp((bioavailability_prediction * 0.35) + (target_receptor_binding * 0.25) + (protein_interaction * 0.2) + (100 - cytotoxicity_risk) * 0.2), 1)
    predictive_toxicity_score = round(clamp(cytotoxicity_risk * 0.7 + model_head_risk * 0.3), 1)
    multi_criteria_screening_score = round(clamp(predicted_drug_efficacy * 0.65 + (100 - predictive_toxicity_score) * 0.35), 1)
    docking_affinity = binding_affinity
    synthesis_yield = round(clamp(weighted_score + (random.random() * 8 - 4)), 1)
    pharmacodynamics_index = round(clamp((predicted_drug_efficacy + protein_interaction + target_receptor_binding) / 3), 1)
    automated_decision = "Promising Candidate" if multi_criteria_screening_score >= 72 else "Needs Optimization" if multi_criteria_screening_score >= 52 else "Reject"

    return {
        "nucleiCount": nuclei_count,
        "meanArea": mean_area,
        "stdArea": std_area,
        "circularity": circularity,
        "aggregationScore": aggregation_score,
        "diceScore": dice_score,
        "iouScore": iou_score,
        "densityPerUnit": density_per_unit,
        "stabilityScore": stability_score,
        "uniformityScore": uniformity_score,
        "interactionStrength": interaction_strength,
        "screeningDecision": screening_decision,
        "particleSizes": particle_sizes,
        "densityData": density_data,
        "radarData": [],
        "dynamicsData": dynamics_data,
        "screeningMetrics": {
            "riskScore": risk_score,
            "multiFactorScore": multi_factor_score,
            "areaComparisonScore": area_comparison_score,
            "aggregationDetectionScore": aggregation_detection_score,
            "psnr": psnr,
            "ssim": ssim,
            "structuralClarity": structural_clarity,
            "segmentationConfidence": segmentation_confidence,
            "membraneInteractionScore": membrane_interaction_score,
            "cytotoxicityRisk": cytotoxicity_risk,
            "surfaceStability": surface_stability,
            "zetaPotentialProxy": zeta_potential_proxy,
            "diffusionCoefficient": diffusion_coefficient,
            "transportEfficiency": transport_efficiency,
            "bioavailabilityPrediction": bioavailability_prediction,
            "structuralConsistency": structural_consistency,
            "clusterFormation": cluster_formation,
            "densityVariation": density_variation,
            "particleOverlap": particle_overlap,
            "stabilityRisk": stability_risk,
            "featureVectorIntegration": feature_vector_integration,
            "weightedScore": weighted_score,
            "thresholdGap": threshold_gap,
            "finalScreeningScore": final_screening_score,
            "modelHeadRisk": model_head_risk,
            "smiles": smiles,
            "molecularWeight": molecular_weight,
            "bindingAffinity": binding_affinity,
            "solubility": solubility,
            "cellUptakeRate": cell_uptake_rate,
            "toxicityLabel": toxicity_label,
            "proteinInteraction": protein_interaction,
            "targetReceptorBinding": target_receptor_binding,
            "diffusionTrend": diffusion_trend,
            "movementTrend": movement_trend,
            "responseTrend": response_trend,
            "formulationTransportScore": formulation_transport_score,
            "predictedDrugEfficacy": predicted_drug_efficacy,
            "predictiveToxicityScore": predictive_toxicity_score,
            "multiCriteriaScreeningScore": multi_criteria_screening_score,
            "dockingAffinity": docking_affinity,
            "synthesisYield": synthesis_yield,
            "pharmacodynamicsIndex": pharmacodynamics_index,
            "automatedDecision": automated_decision,
        },
    }

def make_history_entry(image_name, seed=None):
    return {
        "id": str(uuid.uuid4()),
        "createdAt": datetime.utcnow().isoformat() + "Z",
        "imageName": image_name,
        "imageData": "",
        "result": run_mock_analysis(seed=seed),
    }


In [ ]:

#@title 4) Screening section definitions (same as src/pages/Screening.tsx)
SECTIONS = [
  ("overview", "Screening Module Overview"),
  ("morphology", "Morphology-Based Screening"),
  ("reconstruction", "Reconstruction Quality Assessment"),
  ("interaction", "Nano–Bio Interaction Indicators"),
  ("formulation", "Formulation Stability Evaluation"),
  ("drug", "Drug Prediction (Multimodal)"),
  ("aggregation", "Aggregation Behavior Analysis"),
  ("risk", "Multi-Factor Risk Score Calculation"),
  ("classification", "Screening Decision Classification"),
  ("model", "Model-Based Screening Head (NanoVisionNet-X)"),
  ("visualization", "Screening Output Visualization"),
]

def section_row(section_key, sample):
    s = sample["result"]
    m = s["screeningMetrics"]
    if section_key == "overview":
        return f"{m['riskScore']:.1f} risk | {s['screeningDecision']}"
    if section_key == "morphology":
        return f"{s['nucleiCount']} particles · {s['meanArea']:.1f} area · {s['circularity']:.2f} circ"
    if section_key == "reconstruction":
        return f"PSNR {m['psnr']:.2f} · SSIM {m['ssim']:.3f} · Conf {m['segmentationConfidence']:.1f}"
    if section_key == "interaction":
        return f"Mem {m['membraneInteractionScore']:.1f} · Cyto {m['cytotoxicityRisk']:.1f} · Zeta {m['zetaPotentialProxy']:.1f}"
    if section_key == "formulation":
        return f"Diff {m['diffusionCoefficient']:.3f} · Transport {m['transportEfficiency']:.1f} · Bio {m['bioavailabilityPrediction']:.1f}"
    if section_key == "drug":
        return f"Efficacy {m['predictedDrugEfficacy']:.1f} · Toxicity {m['predictiveToxicityScore']:.1f} · {m['automatedDecision']}"
    if section_key == "aggregation":
        return f"Cluster {m['clusterFormation']:.1f} · Density Δ {m['densityVariation']:.1f} · Overlap {m['particleOverlap']:.1f}"
    if section_key == "risk":
        return f"Vector {m['featureVectorIntegration']:.1f} · Weighted {m['weightedScore']:.1f} · Gap {m['thresholdGap']:.1f}"
    if section_key == "classification":
        return f"{s['screeningDecision']} · Stability Risk {m['stabilityRisk']:.1f}"
    if section_key == "model":
        return f"Encoder {m['featureVectorIntegration']:.1f} · Sigmoid risk {m['modelHeadRisk']:.1f}"
    return f"Risk {m['riskScore']:.1f} · Final {m['finalScreeningScore']:.1f} · PSNR {m['psnr']:.2f}"


In [ ]:

#@title 5) Dashboard state + controls
state = {
    "history": [],
    "search": "",
    "selected_ids": [],
    "open_section": "overview",
}

history_upload = widgets.FileUpload(accept='.json', multiple=False, description='Upload History JSON')
mock_count = widgets.IntSlider(value=6, min=1, max=30, step=1, description='Mock samples')
btn_gen = widgets.Button(description='Generate Mock History', button_style='info', icon='random')
btn_export = widgets.Button(description='Export History JSON', button_style='success', icon='download')

search_input = widgets.Text(description='Search', placeholder='Search sample name')
selection = widgets.SelectMultiple(options=[], description='Samples', rows=10, layout=widgets.Layout(width='60%'))
btn_select_all = widgets.Button(description='Select All', icon='check-square')
btn_clear_sel = widgets.Button(description='Clear Selection', icon='square-o')
btn_delete_selected = widgets.Button(description='Delete Selected', button_style='warning', icon='trash')

section_dropdown = widgets.Dropdown(options=[(title, key) for key, title in SECTIONS], value='overview', description='Section')
status = widgets.HTML('<b>Status:</b> Ready.')
out_summary = widgets.Output()
out_chart = widgets.Output()
out_section = widgets.Output()


def refresh_options():
    q = state['search'].strip().lower()
    filtered = [e for e in state['history'] if q in e['imageName'].lower()]
    options = [(f"{e['imageName']} | {e['createdAt'][:10]}", e['id']) for e in filtered]
    selection.options = options

    existing = {v for _, v in options}
    kept = [sid for sid in state['selected_ids'] if sid in existing]
    if not kept and options:
        kept = [v for _, v in options]
    state['selected_ids'] = kept
    selection.value = tuple(kept)


def selected_samples():
    sel = set(state['selected_ids'])
    return [e for e in state['history'] if e['id'] in sel]


def render_summary_list():
    with out_summary:
        clear_output(wait=True)
        if not state['history']:
            display(Markdown('### Drug Screening Dashboard
No history found. Generate mock samples or upload a history JSON first.'))
            return

        display(Markdown(f"### Drug Screening Dashboard
History samples: **{len(state['history'])}** | Selected: **{len(selected_samples())}**"))
        rows = []
        for e in state['history']:
            m = e['result']['screeningMetrics']
            rows.append({
                'Sample': e['imageName'],
                'Date': e['createdAt'][:10],
                'Risk': round(m['riskScore'], 1),
                'Final': round(m['finalScreeningScore'], 1),
                'PSNR': round(m['psnr'], 2),
                'Decision': e['result']['screeningDecision'],
            })
        display(pd.DataFrame(rows))


def render_chart():
    with out_chart:
        clear_output(wait=True)
        samples = selected_samples()
        if not samples:
            display(Markdown('No selected samples to compare.'))
            return

        labels = [s['imageName'][:16] for s in samples]
        risk = [s['result']['screeningMetrics']['riskScore'] for s in samples]
        final = [s['result']['screeningMetrics']['finalScreeningScore'] for s in samples]
        psnr = [s['result']['screeningMetrics']['psnr'] for s in samples]

        x = np.arange(len(labels))
        w = 0.25
        plt.figure(figsize=(max(8, len(labels)*1.3), 3.8))
        plt.bar(x - w, risk, w, label='Risk')
        plt.bar(x, final, w, label='Final')
        plt.bar(x + w, psnr, w, label='PSNR')
        plt.xticks(x, labels, rotation=25, ha='right')
        plt.title('Risk / Final / PSNR Comparison')
        plt.legend()
        plt.grid(axis='y', alpha=.25)
        plt.tight_layout()
        plt.show()


def render_section():
    with out_section:
        clear_output(wait=True)
        samples = selected_samples()
        key = state['open_section']
        title = dict(SECTIONS).get(key, 'Screening Details')
        display(Markdown(f"### {title}
Showing {len(samples)} selected sample(s)."))
        if not samples:
            display(Markdown('No selected samples.'))
            return
        for s in samples:
            msg = section_row(key, s)
            display(HTML(f"<div style='border:1px solid #ddd;border-radius:8px;padding:10px;margin:8px 0'><b>{s['imageName']}</b><br><span style='color:#666'>{msg}</span></div>"))


def rerender_all():
    refresh_options()
    render_summary_list()
    render_chart()
    render_section()


def on_generate(_):
    n = int(mock_count.value)
    state['history'] = [make_history_entry(f"sample_{i+1:03d}.png", seed=100+i) for i in range(n)]
    state['selected_ids'] = [e['id'] for e in state['history']]
    status.value = f"<b>Status:</b> Generated {n} mock screening samples."
    rerender_all()


def on_export(_):
    payload = json.dumps(state['history'], indent=2)
    with open('nanovision_history.json', 'w', encoding='utf-8') as f:
        f.write(payload)
    status.value = '<b>Status:</b> History exported to nanovision_history.json.'
    if IN_COLAB:
        files.download('nanovision_history.json')


def on_upload(change):
    if not history_upload.value:
        return
    file_info = next(iter(history_upload.value.values()))
    content = file_info['content']
    try:
        parsed = json.loads(content.decode('utf-8'))
        if not isinstance(parsed, list):
            raise ValueError('Uploaded JSON must be a list of history entries.')
        state['history'] = parsed
        state['selected_ids'] = [e.get('id') for e in parsed if e.get('id')]
        status.value = f"<b>Status:</b> Loaded {len(state['history'])} history entries from upload."
    except Exception as e:
        status.value = f"<b>Status:</b> Upload parse failed: {e}"
    rerender_all()


def on_search(change):
    state['search'] = search_input.value
    rerender_all()


def on_select_change(change):
    state['selected_ids'] = list(selection.value)
    render_chart(); render_section(); render_summary_list()


def on_select_all(_):
    state['selected_ids'] = [sid for _, sid in selection.options]
    selection.value = tuple(state['selected_ids'])
    render_chart(); render_section(); render_summary_list()


def on_clear_selection(_):
    state['selected_ids'] = []
    selection.value = tuple()
    render_chart(); render_section(); render_summary_list()


def on_delete_selected(_):
    sel = set(state['selected_ids'])
    if not sel:
        status.value = '<b>Status:</b> No selected samples to delete.'
        return
    before = len(state['history'])
    state['history'] = [e for e in state['history'] if e['id'] not in sel]
    removed = before - len(state['history'])
    state['selected_ids'] = [e['id'] for e in state['history']]
    status.value = f"<b>Status:</b> Deleted {removed} selected sample(s)."
    rerender_all()


def on_section_change(change):
    state['open_section'] = section_dropdown.value
    render_section()


btn_gen.on_click(on_generate)
btn_export.on_click(on_export)
history_upload.observe(on_upload, names='value')
search_input.observe(on_search, names='value')
selection.observe(on_select_change, names='value')
btn_select_all.on_click(on_select_all)
btn_clear_sel.on_click(on_clear_selection)
btn_delete_selected.on_click(on_delete_selected)
section_dropdown.observe(on_section_change, names='value')


In [ ]:

#@title 6) Launch dashboard
ui = widgets.VBox([
    widgets.HTML('<h2>Drug Screening Dashboard</h2><p>Select history samples and open any screening section details.</p>'),
    widgets.HBox([history_upload, mock_count, btn_gen, btn_export]),
    status,
    widgets.HBox([search_input, selection]),
    widgets.HBox([btn_select_all, btn_clear_sel, btn_delete_selected]),
    out_summary,
    widgets.HTML('<hr><h4>Comparison Chart</h4>'),
    out_chart,
    widgets.HTML('<hr><h4>Section Drill-Down</h4>'),
    section_dropdown,
    out_section,
])

display(ui)
rerender_all()
